# Data & Feature Versioning: DVC, Feature Stores & Point-in-Time Joins

Robust ML pipelines require explicit data and feature versioning guarantees:
1. **Content-Addressable Data Versioning (DVC Pattern)**: Tracking large datasets with cryptographic hash pointers without bloating Git.
2. **Feature Store Architecture (Feast Pattern)**: Dual-storage abstraction (Low-latency Online KV Store vs Historical Offline Lakehouse).
3. **Point-in-Time Correctness (AS-OF Joins)**: Preventing target leakage during training set generation by matching observations with features recorded strictly before the event.
4. **Data Lineage DAG**: Tracking hash dependencies to execute only invalidated processing steps.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('07-mlops/data-versioning/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import data_registry
print('data_registry loaded')

## 1. Content-Addressable Data Tracking (DVC Pattern)
Lightweight `.dvc` metadata files commit to Git while actual data payloads reside in cloud storage (S3/GCS).

In [ ]:
from data_registry import DataVersionControl

dvc = DataVersionControl()

# Raw training partition
raw_csv_v1 = b"user_id,amount,timestamp\nu1,24.50,1690000000\nu2,120.00,1690000500"
manifest_v1 = dvc.track_file("data/raw/credit_transactions.csv", raw_csv_v1)

print("Generated DVC Manifest:")
for k, v in manifest_v1.to_dict().items():
    print(f"  {k:12s}: {v}")

# Integrity check
print("Integrity check on clean data :", dvc.verify_integrity(manifest_v1, raw_csv_v1))
print("Integrity check on tampered data:", dvc.verify_integrity(manifest_v1, raw_csv_v1 + b"\nu3,999.00"))

## 2. Online Feature Store for Real-Time Inference
Ultra-fast sub-10ms key-value lookups for production model scoring.

In [ ]:
from data_registry import OnlineFeatureStore

online_store = OnlineFeatureStore()

# Ingest computed features into online store
online_store.write_features("usr_101", {"avg_daily_spend": 82.4, "failed_logins_24h": 0})
online_store.write_features("usr_102", {"avg_daily_spend": 415.0, "failed_logins_24h": 3})

# Real-time inference request needs features for usr_101 and usr_102
retrieved_features = online_store.get_online_features(["usr_101", "usr_102"])
for i, feat in enumerate(retrieved_features, 101):
    print(f"Features for usr_{i}: {feat}")

## 3. Offline Feature Store: Point-in-Time Correctness (Preventing Data Leakage)
Joining training observations with historical feature logs without looking into the future.

In [ ]:
from data_registry import OfflineFeatureStore

offline_store = OfflineFeatureStore()

# Historical feature updates for customer 'u1'
offline_store.log_features("u1", timestamp=100.0, features={"risk_score": 0.12})
offline_store.log_features("u1", timestamp=200.0, features={"risk_score": 0.45})
offline_store.log_features("u1", timestamp=300.0, features={"risk_score": 0.88})

# Training observations (e.g. loan defaults or fraud events)
observations = [
    {"user_id": "u1", "timestamp": 150.0, "event": "loan_application_1"},
    {"user_id": "u1", "timestamp": 250.0, "event": "loan_application_2"}
]

# AS-OF Join: each observation receives features as they existed at observation time
training_dataset = offline_store.point_in_time_join(observations, entity_key="user_id", timestamp_key="timestamp")

for row in training_dataset:
    print(f"Event at t={row['timestamp']:.0f} matched feature recorded at t={row['feature_timestamp']:.0f} (risk_score={row['risk_score']})")

## 4. Data Lineage & Stale Stage Detection
Verifying hash states across the end-to-end processing graph.

In [ ]:
from data_registry import PipelineStage, DataLineageDAG

dag = DataLineageDAG()

# Stage 1: Feature Engineering
dag.register_stage(PipelineStage(
    name="features",
    inputs={"data/raw/transactions.csv": manifest_v1.content_hash},
    outputs={"data/features/features.parquet": "hash_features_abc"}
))

# Check with unchanged inputs
print("Is 'features' stage stale (unchanged data)?", dag.is_stage_stale("features", {"data/raw/transactions.csv": manifest_v1.content_hash}))

# Check when new transactions arrive
new_data_hash = "sha256_new_incoming_batch_def"
print("Is 'features' stage stale (new incoming batch)?", dag.is_stage_stale("features", {"data/raw/transactions.csv": new_data_hash}))